# rPPG · 三步跑完

| 格 | 干什么 | 什么时候要改 |
|---|---|---|
| 第 0 格 | 环境：挂 Drive → 拉代码（钉死版本）→ 判新 VM → 装依赖/补丁 → 凭据 → 数据视图 | **永不改**；换机器/换硬件时重跑一次即可 |
| 第 1 格 | 跑实验：`run("名字", ctx)` | 只改实验名和参数 |
| 第 2 格 | 汇总：所有日志里的 FFT 指标 + 版本戳 | 不用改 |

所有实验都定义在 `projects/rppg/experiments.py` 里，不在 notebook 里。
原因：CPU 调试和 GPU 复跑是**两台不同的 VM**，内核变量不会跟着走 ——
"调通的成果"必须落成进 git 的代码 + 进 Drive 的日志，才能复用。


In [ ]:
# ============ 第 0 格：环境（固定块，永远不要改）============
# 幂等：挂 Drive → 拉代码 → 判新 VM → 装依赖/打补丁 → 载凭据 → 备数据
import os, subprocess, sys
from pathlib import Path

repo = Path("/content/colab-workbench")
if not repo.exists():
    subprocess.run(["git", "clone", "-q",
        "https://github.com/Elisabethhui/colab-workbench.git",
        str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=False)

# 同一个内核里重跑时，必须丢掉旧的模块缓存，否则改动不生效
for _m in [m for m in sys.modules if m == "bootstrap" or m.startswith("projects")]:
    del sys.modules[_m]
sys.path.insert(0, str(repo))

from bootstrap import setup
ctx = setup("rppg")

## 第 1 格 · 跑实验

实验名固定，一个名字 = 一份配置 + 一段日志：

| 名字 | 硬件 | 预期 | 说明 |
|---|---|---|---|
| `baseline_classical` | CPU | 8–20 分钟 | POS/CHROM 古典基线，不需要权重，**CPU 就能出真指标** |
| `effphys_pure` | GPU | ~6 分钟 | PURE 训的 EfficientPhys → UBFC 全部 42 个 |
| `effphys_scamps` | GPU | ~6 分钟 | SCAMPS 训的 EfficientPhys → UBFC 全部 42 个 |
| `train_e1` | GPU | 20–40 分钟 | 自己训练：UBFC 前 80% 训 / 后 20% 测（30 epoch） |
| `author_test20` | GPU | ~2 分钟 | 作者预训练模型只跑后 20%，和 E1 同区间对照 |

三个开关：

* `limit=2` —— 只跑前 2 个 subject（冒烟，1~2 分钟）
* `dry_run=True` —— 只生成配置并打印关键字段，不执行（CPU 阶段验证用）
* `force=True` —— 硬件不够也硬跑（GPU 实验在 CPU 上会非常慢）

当前内核没有 GPU 时，GPU 实验会**直接跳过并提示**，不会白跑 40 分钟。


In [ ]:
import sys
from pathlib import Path

# experiments.py 在仓库里（跟着第 0 格拉下来的那份代码），不在 notebook 里
sys.path.insert(0, str(Path(ctx.paths.repo) / "projects" / "rppg"))
from experiments import run, summary, list_experiments

list_experiments()                      # 先看有什么、各要什么硬件

run("baseline_classical", ctx)          # ← 改这里。冒烟就加 limit=2，验配置就加 dry_run=True


## 第 2 格 · 汇总

把所有日志（在 Drive 上的 `logs/`，换机器不丢）里的 FFT 指标拉成一张表，
并带上版本戳：哪个上游 commit + 什么硬件跑出来的。换机器后重跑第 0 格再来这格，就能看到历史。


In [ ]:
summary(ctx)